In [2]:
import os
import urllib.request
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

data_dir = 'data'
fig_dir = 'figures'
for d in [data_dir, fig_dir]:
    os.makedirs(d, exist_ok=True)

In [4]:
# Data Download and Processing
base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
files_to_download = ["DEMO_L.xpt", "BMX_L.xpt", "BPXO_L.xpt", "SLQ_L.xpt", "PAQ_L.xpt"]

for file_name in files_to_download:
    file_path = os.path.join(data_dir, file_name)
    if not os.path.exists(file_path):
        urllib.request.urlretrieve(base_url + file_name, file_path)

demo = pd.read_sas(f'{data_dir}/DEMO_L.xpt')
bmx  = pd.read_sas(f'{data_dir}/BMX_L.xpt')
bpxo = pd.read_sas(f'{data_dir}/BPXO_L.xpt')
slq  = pd.read_sas(f'{data_dir}/SLQ_L.xpt')
paq  = pd.read_sas(f'{data_dir}/PAQ_L.xpt')

df = demo.merge(bmx, on='SEQN', how='inner')
df = df.merge(bpxo, on='SEQN', how='inner')
df = df.merge(slq, on='SEQN', how='inner')
df = df.merge(paq, on='SEQN', how='inner')

columns_to_keep = [
    'SEQN', 'RIDAGEYR', 'RIAGENDR', 'WTMEC2YR',
    'BMXBMI', 'BPXOSY1', 'SLD012', 'PAD680'
]
df = df[columns_to_keep].copy()
df['PAD680'] = df['PAD680'].replace([7777, 9999], np.nan)

df = df.rename(columns={
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'WTMEC2YR': 'Weight',
    'BMXBMI': 'BMI',
    'BPXOSY1': 'Systolic_BP',
    'SLD012': 'Sleep_Hours',
    'PAD680': 'Sedentary_Min'
})
df['Gender'] = df['Gender'].map({1.0: 'Male', 2.0: 'Female'})

df_adults = df[df['Age'] >= 20].copy()
df_final = df_adults.dropna(subset=['Systolic_BP', 'BMI', 'Sleep_Hours', 'Sedentary_Min']).copy()
df_final = df_final.reset_index(drop=True)

N = len(df_final)
print(f"\nFinal Analytical Sample Size: N = {N}")



Final Analytical Sample Size: N = 5714


In [5]:
## 1. DEFINE EVENTS AND RANDOM VARIABLES (common task 1)

# Event H: High Systolic Blood Pressure (Stage 1+ Hypertension threshold: SBP >= 130 mmHg)
df_final['Event_H_HighBP'] = (df_final['Systolic_BP'] >= 130).astype(int)

# Event O: Obesity (Clinical threshold: BMI >= 30.0 kg/m^2)
df_final['Event_O_Obese'] = (df_final['BMI'] >= 30.0).astype(int)

# Event S: Short / Inadequate Sleep (Clinical guideline: Sleep_Hours < 7.0 hours)
df_final['Event_S_ShortSleep'] = (df_final['Sleep_Hours'] < 7.0).astype(int)

# Event D: High Sedentary Behavior (Sitting >= 480 minutes/day = 8 hours/day)
df_final['Event_D_HighSed'] = (df_final['Sedentary_Min'] >= 480).astype(int)

# Subgroup variables for Path A.1
df_final['Age_Group'] = pd.cut(
    df_final['Age'],
    bins=[19, 39, 59, 85],
    labels=['20-39 (Young)', '40-59 (Middle)', '60+ (Older)']
)
df_final['BMI_Category'] = pd.cut(
    df_final['BMI'],
    bins=[0, 25.0, 30.0, 100.0],
    right=False,
    labels=['Normal/Under (<25)', 'Overweight (25-29.9)', 'Obese (>=30)']
)

In [6]:
## 2. MARGINAL, JOINT, AND CONDITIONAL PROBABILITIES (TASKS 2, 3, 4 & PATH A.1)

p_H = df_final['Event_H_HighBP'].mean()
p_O = df_final['Event_O_Obese'].mean()
p_S = df_final['Event_S_ShortSleep'].mean()
p_D = df_final['Event_D_HighSed'].mean()

n_H = int(df_final['Event_H_HighBP'].sum())
n_O = int(df_final['Event_O_Obese'].sum())
n_S = int(df_final['Event_S_ShortSleep'].sum())
n_D = int(df_final['Event_D_HighSed'].sum())

print("\n--- 1. Marginal Probabilities ---")
print(f"P(H: SBP >= 130)        = {p_H:.4f} ({n_H}/{N})")
print(f"P(O: BMI >= 30)         = {p_O:.4f} ({n_O}/{N})")
print(f"P(S: Sleep < 7h)        = {p_S:.4f} ({n_S}/{N})")
print(f"P(D: Sedentary >= 480m) = {p_D:.4f} ({n_D}/{N})")


def analyze_event_pair(df_in, col_A, col_B, name_A, name_B):
    N_tot = len(df_in)
    pA = df_in[col_A].mean()
    pB = df_in[col_B].mean()
    n_AB = int(((df_in[col_A] == 1) & (df_in[col_B] == 1)).sum())
    p_AB = n_AB / N_tot
    p_A_prod_B = pA * pB
    p_A_given_B = df_in.loc[df_in[col_B] == 1, col_A].mean()
    p_A_given_notB = df_in.loc[df_in[col_B] == 0, col_A].mean()
    p_B_given_A = df_in.loc[df_in[col_A] == 1, col_B].mean()
    p_B_given_notA = df_in.loc[df_in[col_A] == 0, col_B].mean()
    diff_indep = p_AB - p_A_prod_B
    rr = p_A_given_B / p_A_given_notB
    return {
        'name_A': name_A, 'name_B': name_B,
        'pA': pA, 'pB': pB, 'n_AB': n_AB, 'p_AB': p_AB,
        'p_A_prod_B': p_A_prod_B, 'diff_indep': diff_indep,
        'p_A_given_B': p_A_given_B, 'p_A_given_notB': p_A_given_notB,
        'p_B_given_A': p_B_given_A, 'p_B_given_notA': p_B_given_notA,
        'rr': rr
    }



pair_HO = analyze_event_pair(df_final, 'Event_H_HighBP', 'Event_O_Obese', 'High BP (H)', 'Obesity (O)')
pair_HS = analyze_event_pair(df_final, 'Event_H_HighBP', 'Event_S_ShortSleep', 'High BP (H)', 'Short Sleep (S)')
pair_HD = analyze_event_pair(df_final, 'Event_H_HighBP', 'Event_D_HighSed', 'High BP (H)', 'High Sedentary (D)')

print("\n--- 2. Joint, Conditional, and Independence Analysis ---")
for res in [pair_HO, pair_HS, pair_HD]:
    print(f"Pair: {res['name_A']} & {res['name_B']}")
    print(f"  Joint P(A n B) = {res['p_AB']:.4f} (n={res['n_AB']}) vs P(A)*P(B) = {res['p_A_prod_B']:.4f} | Diff = {res['diff_indep']:+.4f}")
    print(f"  Conditional P(A | B) = {res['p_A_given_B']:.4f} vs P(A | B^c) = {res['p_A_given_notB']:.4f} | Ratio = {res['rr']:.2f}")


# Path A.1: Subgroup conditional probabilities (Gender, Age_Group, BMI_Category)
p_H_male = df_final.loc[df_final['Gender'] == 'Male', 'Event_H_HighBP'].mean()
p_H_female = df_final.loc[df_final['Gender'] == 'Female', 'Event_H_HighBP'].mean()

age_grp_probs = df_final.groupby('Age_Group', observed=False)['Event_H_HighBP'].agg(['count', 'mean'])
bmi_grp_probs = df_final.groupby('BMI_Category', observed=False)['Event_H_HighBP'].agg(['count', 'mean'])

print("\n--- Path A.1: Conditional Probability of High BP in Subgroups ---")
print(f"P(H | Male)   = {p_H_male:.4f}")
print(f"P(H | Female) = {p_H_female:.4f}")
print("\nBy Age Group:\n", age_grp_probs)
print("\nBy BMI Category:\n", bmi_grp_probs)



--- 1. Marginal Probabilities ---
P(H: SBP >= 130)        = 0.3122 (1784/5714)
P(O: BMI >= 30)         = 0.4095 (2340/5714)
P(S: Sleep < 7h)        = 0.2107 (1204/5714)
P(D: Sedentary >= 480m) = 0.3306 (1889/5714)

--- 2. Joint, Conditional, and Independence Analysis ---
Pair: High BP (H) & Obesity (O)
  Joint P(A n B) = 0.1334 (n=762) vs P(A)*P(B) = 0.1279 | Diff = +0.0055
  Conditional P(A | B) = 0.3256 vs P(A | B^c) = 0.3029 | Ratio = 1.08
Pair: High BP (H) & Short Sleep (S)
  Joint P(A n B) = 0.0660 (n=377) vs P(A)*P(B) = 0.0658 | Diff = +0.0002
  Conditional P(A | B) = 0.3131 vs P(A | B^c) = 0.3120 | Ratio = 1.00
Pair: High BP (H) & High Sedentary (D)
  Joint P(A n B) = 0.0907 (n=518) vs P(A)*P(B) = 0.1032 | Diff = -0.0126
  Conditional P(A | B) = 0.2742 vs P(A | B^c) = 0.3310 | Ratio = 0.83

--- Path A.1: Conditional Probability of High BP in Subgroups ---
P(H | Male)   = 0.3481
P(H | Female) = 0.2829

By Age Group:
                 count      mean
Age_Group                     

In [7]:
## 3. BAYES THEOREM (COMMON TASK 9)

prior_H = p_H
lik_O_given_H = pair_HO['p_B_given_A']
lik_O_given_notH = pair_HO['p_B_given_notA']
marginal_O_total_prob = lik_O_given_H * prior_H + lik_O_given_notH * (1 - prior_H)
post_H_given_O = (lik_O_given_H * prior_H) / marginal_O_total_prob

print("\n--- 3. Bayes' Theorem ---")
print(f"Case 1 (Given Obesity O): Prior P(H)={prior_H:.4f}, P(O|H)={lik_O_given_H:.4f}, P(O|H^c)={lik_O_given_notH:.4f} -> Posterior P(H|O)={post_H_given_O:.4f}")



--- 3. Bayes' Theorem ---
Case 1 (Given Obesity O): Prior P(H)=0.3122, P(O|H)=0.4271, P(O|H^c)=0.4015 -> Posterior P(H|O)=0.3256


In [9]:
## 4. DISCRETE DISTRIBUTION & BINOMIAL MODEL (TASK 5 & PATH A.2)

n_trials = 25
M_subsamples = N // n_trials

# (a) Natural order of dataset
subsamples_natural = df_final['Event_H_HighBP'].iloc[:M_subsamples * n_trials].values.reshape(M_subsamples, n_trials)
counts_natural = subsamples_natural.sum(axis=1)

# (b) Randomly permuted order
np.random.seed(42)
shuffled_idx = np.random.permutation(N)
subsamples_random = df_final['Event_H_HighBP'].iloc[shuffled_idx[:M_subsamples * n_trials]].values.reshape(M_subsamples, n_trials)
counts_random = subsamples_random.sum(axis=1)

# Theoretical Binomial(n=25, p=p_H)
theo_binom_mean = n_trials * p_H
theo_binom_var  = n_trials * p_H * (1 - p_H)
emp_rand_mean   = counts_random.mean()
emp_rand_var    = counts_random.var(ddof=1)
emp_nat_mean    = counts_natural.mean()
emp_nat_var     = counts_natural.var(ddof=1)

print("\n--- 4. Binomial Model Comparison (n = 25, M = 228 subsamples) ---")
print(f"Theoretical Binomial(25, {p_H:.4f}): Mean = {theo_binom_mean:.3f}, Variance = {theo_binom_var:.3f}")
print(f"Empirical Random Subsamples:        Mean = {emp_rand_mean:.3f}, Variance = {emp_rand_var:.3f}")
print(f"Empirical Natural-Order Subsamples: Mean = {emp_nat_mean:.3f}, Variance = {emp_nat_var:.3f}")


--- 4. Binomial Model Comparison (n = 25, M = 228 subsamples) ---
Theoretical Binomial(25, 0.3122): Mean = 7.805, Variance = 5.368
Empirical Random Subsamples:        Mean = 7.816, Variance = 5.305
Empirical Natural-Order Subsamples: Mean = 7.803, Variance = 5.613


In [10]:
## 5. CONTINUOUS VARIABLES & NORMAL APPROXIMATION (TASK 6 & PATH A.3)


bp_mean = df_final['Systolic_BP'].mean()
bp_sd   = df_final['Systolic_BP'].std(ddof=1)
bp_var  = df_final['Systolic_BP'].var(ddof=1)
bp_skew = stats.skew(df_final['Systolic_BP'])
bp_kurt = stats.kurtosis(df_final['Systolic_BP']) # excess kurtosis

bmi_mean = df_final['BMI'].mean()
bmi_sd   = df_final['BMI'].std(ddof=1)
bmi_var  = df_final['BMI'].var(ddof=1)
bmi_skew = stats.skew(df_final['BMI'])
bmi_kurt = stats.kurtosis(df_final['BMI'])

# Compare empirical intervals vs Normal(bp_mean, bp_sd)
emp_1sd = ((df_final['Systolic_BP'] >= bp_mean - bp_sd) & (df_final['Systolic_BP'] <= bp_mean + bp_sd)).mean()
emp_2sd = ((df_final['Systolic_BP'] >= bp_mean - 2*bp_sd) & (df_final['Systolic_BP'] <= bp_mean + 2*bp_sd)).mean()
theo_1sd = stats.norm.cdf(1) - stats.norm.cdf(-1)
theo_2sd = stats.norm.cdf(2) - stats.norm.cdf(-2)

emp_ge130  = (df_final['Systolic_BP'] >= 130).mean()
theo_ge130 = 1 - stats.norm.cdf(130, loc=bp_mean, scale=bp_sd)

emp_ge160  = (df_final['Systolic_BP'] >= 160).mean()
theo_ge160 = 1 - stats.norm.cdf(160, loc=bp_mean, scale=bp_sd)

print("\n--- 5. Normal Approximation for Systolic BP ---")
print(f"Systolic BP: Mean={bp_mean:.2f}, SD={bp_sd:.2f}, Skew={bp_skew:.2f}, Excess Kurtosis={bp_kurt:.2f}")
print(f"Within +-1 SD: Empirical = {emp_1sd:.4f} vs Normal = {theo_1sd:.4f}")
print(f"Within +-2 SD: Empirical = {emp_2sd:.4f} vs Normal = {theo_2sd:.4f}")
print(f"P(SBP >= 130): Empirical = {emp_ge130:.4f} vs Normal = {theo_ge130:.4f}")
print(f"P(SBP >= 160): Empirical = {emp_ge160:.4f} vs Normal = {theo_ge160:.4f}")


--- 5. Normal Approximation for Systolic BP ---
Systolic BP: Mean=123.00, SD=18.47, Skew=0.91, Excess Kurtosis=1.70
Within +-1 SD: Empirical = 0.7259 vs Normal = 0.6827
Within +-2 SD: Empirical = 0.9538 vs Normal = 0.9545
P(SBP >= 130): Empirical = 0.3122 vs Normal = 0.3523
P(SBP >= 160): Empirical = 0.0397 vs Normal = 0.0226


In [11]:
## 6. EXPECTED VALUE, VARIANCE, AND COVARIANCE (TASKS 7 & 8)

quant_cols = ['Systolic_BP', 'BMI', 'Sleep_Hours', 'Sedentary_Min', 'Age']
ev_var_df = pd.DataFrame({
    'Expected_Value_Mean': df_final[quant_cols].mean(),
    'Variance': df_final[quant_cols].var(ddof=1),
    'Std_Dev': df_final[quant_cols].std(ddof=1),
    'Skewness': df_final[quant_cols].apply(stats.skew)
})
print("\n--- 6. Expected Value and Variance ---")
print(ev_var_df.round(2))

cov_matrix = df_final[quant_cols].cov()
cov_bp_bmi   = cov_matrix.loc['Systolic_BP', 'BMI']
cov_bp_sleep = cov_matrix.loc['Systolic_BP', 'Sleep_Hours']
cov_bp_sed   = cov_matrix.loc['Systolic_BP', 'Sedentary_Min']
cov_bp_age   = cov_matrix.loc['Systolic_BP', 'Age']
cov_bmi_sed  = cov_matrix.loc['BMI', 'Sedentary_Min']

print("\n--- 7. Covariances with Systolic_BP ---")
print(f"Cov(Systolic_BP, BMI)           = {cov_bp_bmi:+.2f} mmHg * (kg/m^2)")
print(f"Cov(Systolic_BP, Age)           = {cov_bp_age:+.2f} mmHg * years")
print(f"Cov(Systolic_BP, Sleep_Hours)   = {cov_bp_sleep:+.2f} mmHg * hours")
print(f"Cov(Systolic_BP, Sedentary_Min) = {cov_bp_sed:+.2f} mmHg * minutes")
print(f"Cov(BMI, Sedentary_Min)         = {cov_bmi_sed:+.2f} (kg/m^2) * minutes")



--- 6. Expected Value and Variance ---
               Expected_Value_Mean  Variance  Std_Dev  Skewness
Systolic_BP                 123.00    341.07    18.47      0.91
BMI                          29.74     51.27     7.16      1.11
Sleep_Hours                   7.70      2.48     1.57     -0.08
Sedentary_Min               363.94  43603.19   208.81      0.80
Age                          53.75    292.00    17.09     -0.28

--- 7. Covariances with Systolic_BP ---
Cov(Systolic_BP, BMI)           = +2.85 mmHg * (kg/m^2)
Cov(Systolic_BP, Age)           = +120.92 mmHg * years
Cov(Systolic_BP, Sleep_Hours)   = +0.32 mmHg * hours
Cov(Systolic_BP, Sedentary_Min) = -204.96 mmHg * minutes
Cov(BMI, Sedentary_Min)         = +179.92 (kg/m^2) * minutes


In [12]:
## 7. ROLE OF SAMPLING WEIGHTS IN EMPIRICAL PROBABILITY (PATH A.4)

w = df_final['Weight'].values

def weighted_prob(series, weights):
    return np.sum(series * weights) / np.sum(weights)

w_p_H = weighted_prob(df_final['Event_H_HighBP'], w)
w_p_O = weighted_prob(df_final['Event_O_Obese'], w)
w_p_S = weighted_prob(df_final['Event_S_ShortSleep'], w)
w_p_D = weighted_prob(df_final['Event_D_HighSed'], w)

# Conditional weighted probability P_w(H | O) and P_w(H | O^c)
mask_O = df_final['Event_O_Obese'] == 1
w_p_H_given_O    = weighted_prob(df_final.loc[mask_O, 'Event_H_HighBP'], df_final.loc[mask_O, 'Weight'])
w_p_H_given_notO = weighted_prob(df_final.loc[~mask_O, 'Event_H_HighBP'], df_final.loc[~mask_O, 'Weight'])

print("\n--- 8. Unweighted vs Weighted Empirical Probabilities (Path A.4) ---")
print(f"Event H (SBP >= 130):      Raw = {p_H:.4f} | Weighted = {w_p_H:.4f} | Diff = {p_H - w_p_H:+.4f}")
print(f"Event O (BMI >= 30):       Raw = {p_O:.4f} | Weighted = {w_p_O:.4f} | Diff = {p_O - w_p_O:+.4f}")
print(f"Event S (Sleep < 7h):      Raw = {p_S:.4f} | Weighted = {w_p_S:.4f} | Diff = {p_S - w_p_S:+.4f}")
print(f"Event D (Sedentary >= 8h): Raw = {p_D:.4f} | Weighted = {w_p_D:.4f} | Diff = {p_D - w_p_D:+.4f}")
print(f"Conditional P(H | O):      Raw = {pair_HO['p_A_given_B']:.4f} | Weighted = {w_p_H_given_O:.4f}")



--- 8. Unweighted vs Weighted Empirical Probabilities (Path A.4) ---
Event H (SBP >= 130):      Raw = 0.3122 | Weighted = 0.2701 | Diff = +0.0421
Event O (BMI >= 30):       Raw = 0.4095 | Weighted = 0.4016 | Diff = +0.0079
Event S (Sleep < 7h):      Raw = 0.2107 | Weighted = 0.2067 | Diff = +0.0041
Event D (Sedentary >= 8h): Raw = 0.3306 | Weighted = 0.3351 | Diff = -0.0045
Conditional P(H | O):      Raw = 0.3256 | Weighted = 0.2997


In [13]:
## 8. GENERATE ALL 4 FIGURES FOR PHASE 2 REPORT

# --- FIGURE 1: Marginal, Joint, and Subgroup Conditional Probabilities ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel A: Conditional Probability of High BP given Risk Events
cond_labels = ['Overall\nP(H)', 'Non-Obese\nP(H|O^c)', 'Obese\nP(H|O)',
               'Sleep >=7h\nP(H|S^c)', 'Sleep <7h\nP(H|S)',
               'Sed <8h\nP(H|D^c)', 'Sed >=8h\nP(H|D)']
cond_vals = [p_H, pair_HO['p_A_given_notB'], pair_HO['p_A_given_B'],
             pair_HS['p_A_given_notB'], pair_HS['p_A_given_B'],
             pair_HD['p_A_given_notB'], pair_HD['p_A_given_B']]
colors_a = ['gray', '#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f']
bars = axes[0, 0].bar(cond_labels, cond_vals, color=colors_a, edgecolor='black', alpha=0.85)
axes[0, 0].axhline(p_H, color='red', linestyle='--', linewidth=1.2, label=f'Baseline P(H) = {p_H:.3f}')
axes[0, 0].set_title('A. Conditional Probability of High Systolic BP (SBP >= 130 mmHg)')
axes[0, 0].set_ylabel('Empirical Probability')
axes[0, 0].set_ylim(0, max(cond_vals) * 1.2)
axes[0, 0].legend()
for bar in bars:
    h = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., h + 0.008, f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)

# Panel B: Path A.1 Subgroup Conditional Probabilities by Age Group and Gender
age_gender_df = df_final.groupby(['Age_Group', 'Gender'], observed=False)['Event_H_HighBP'].mean().unstack()
age_gender_df.plot(kind='bar', ax=axes[0, 1], color=['#e78ac3', '#66c2a5'], edgecolor='black', rot=0)
axes[0, 1].set_title('B. P(SBP >= 130 mmHg | Age Group, Gender) [Path A.1]')
axes[0, 1].set_xlabel('Age Group (Years)')
axes[0, 1].set_ylabel('Conditional Probability P(H | Subgroup)')
axes[0, 1].set_ylim(0, 0.65)
axes[0, 1].legend(title='Gender')
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container, fmt='%.3f', padding=3, fontsize=8.5)

# Panel C: Independence Check: Joint P(A n B) vs Product P(A)*P(B)
indep_df = pd.DataFrame({
    'Event Pair': ['High BP & Obesity\n(H n O)', 'High BP & Short Sleep\n(H n S)', 'High BP & High Sedentary\n(H n D)'],
    'Empirical Joint P(A n B)': [pair_HO['p_AB'], pair_HS['p_AB'], pair_HD['p_AB']],
    'Independent Product P(A)P(B)': [pair_HO['p_A_prod_B'], pair_HS['p_A_prod_B'], pair_HD['p_A_prod_B']]
}).set_index('Event Pair')
indep_df.plot(kind='bar', ax=axes[1, 0], color=['#1f78b4', '#a6cee3'], edgecolor='black', rot=0)
axes[1, 0].set_title('C. Independence Evaluation: Joint P(A n B) vs. P(A)P(B)')
axes[1, 0].set_ylabel('Probability')
axes[1, 0].set_ylim(0, max(indep_df.max()) * 1.25)
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt='%.4f', padding=3, fontsize=8.5)

# Panel D: Conditional Probability across BMI Categories
bmi_bars = axes[1, 1].bar(bmi_grp_probs.index.astype(str), bmi_grp_probs['mean'],
                          color=['#a6d854', '#ffd92f', '#fc8d62'], edgecolor='black')
axes[1, 1].axhline(p_H, color='red', linestyle='--', linewidth=1.2, label=f'Marginal P(H) = {p_H:.3f}')
axes[1, 1].set_title('D. Conditional Probability P(SBP >= 130 | BMI Category)')
axes[1, 1].set_xlabel('Clinical BMI Category (kg/m^2)')
axes[1, 1].set_ylabel('Conditional Probability')
axes[1, 1].set_ylim(0, 0.50)
axes[1, 1].legend()
for bar in bmi_bars:
    h = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., h + 0.008, f'{h:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{fig_dir}/phase2_fig1.png')
plt.savefig('phase2_fig1.png')
plt.close()

# --- FIGURE 2: Discrete Distribution & Binomial Model (Task 5 & Path A.2) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
k_vals = np.arange(0, 18)
binom_pmf = stats.binom.pmf(k_vals, n_trials, p_H)

# Panel A: Randomly Shuffled Subsamples vs Theoretical Binomial PMF
emp_counts_rand_freq = pd.Series(counts_random).value_counts(normalize=True).reindex(k_vals, fill_value=0)
axes[0].bar(k_vals - 0.18, emp_counts_rand_freq.values, width=0.36, color='skyblue', edgecolor='black',
            label=f'Empirical Random Subsamples\n(Mean={emp_rand_mean:.2f}, Var={emp_rand_var:.2f})')
axes[0].bar(k_vals + 0.18, binom_pmf, width=0.36, color='salmon', edgecolor='black', alpha=0.85,
            label=f'Theoretical Binomial(n=25, p={p_H:.3f})\n(Mean={theo_binom_mean:.2f}, Var={theo_binom_var:.2f})')
axes[0].set_title('A. Count of High BP (SBP >= 130) in Random Subsamples (n=25, M=228)')
axes[0].set_xlabel('Number of High BP Cases (k) per Subsample of n=25')
axes[0].set_ylabel('Relative Frequency / Probability P(K = k)')
axes[0].set_xticks(k_vals)
axes[0].legend()

# Panel B: Natural Order (Clustered/Ordered) Subsamples vs Theoretical Binomial PMF
emp_counts_nat_freq = pd.Series(counts_natural).value_counts(normalize=True).reindex(k_vals, fill_value=0)
axes[1].bar(k_vals - 0.18, emp_counts_nat_freq.values, width=0.36, color='lightgreen', edgecolor='black',
            label=f'Empirical Natural-Order Subsamples\n(Mean={emp_nat_mean:.2f}, Var={emp_nat_var:.2f})')
axes[1].plot(k_vals, binom_pmf, 'ro-', linewidth=2, markersize=6,
             label=f'Theoretical Binomial PMF\n(Var={theo_binom_var:.2f})')
axes[1].set_title('B. Natural-Order Subsamples vs. Binomial PMF')
axes[1].set_xlabel('Number of High BP Cases (k) per Subsample of n=25')
axes[1].set_ylabel('Relative Frequency / Probability P(K = k)')
axes[1].set_xticks(k_vals)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{fig_dir}/phase2_fig2.png')
plt.savefig('phase2_fig2.png')
plt.close()

# --- FIGURE 3: Continuous Distributions & Normal Approximation (Task 6 & Path A.3) ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel A: Systolic BP Histogram + Fitted Normal PDF
x_bp = np.linspace(df_final['Systolic_BP'].min(), df_final['Systolic_BP'].max(), 300)
pdf_bp = stats.norm.pdf(x_bp, loc=bp_mean, scale=bp_sd)
sns.histplot(df_final['Systolic_BP'], bins=35, stat='density', color='skyblue', edgecolor='black', alpha=0.6, ax=axes[0, 0], label='Empirical Density')
sns.kdeplot(df_final['Systolic_BP'], color='navy', linewidth=2, ax=axes[0, 0], label='Empirical KDE')
axes[0, 0].plot(x_bp, pdf_bp, 'r--', linewidth=2.2, label=f'Normal Fit N({bp_mean:.1f}, {bp_sd:.1f}^2)')
axes[0, 0].axvline(130, color='darkgreen', linestyle=':', linewidth=2, label='Clinical Cutoff (130 mmHg)')
axes[0, 0].set_title(f'A. Systolic BP Distribution vs. Normal Fit (Skew={bp_skew:.2f}, Kurt={bp_kurt:.2f})')
axes[0, 0].set_xlabel('Systolic Blood Pressure (mmHg)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

# Panel B: Normal Q-Q Plot for Systolic BP
stats.probplot(df_final['Systolic_BP'], dist="norm", plot=axes[0, 1])
axes[0, 1].get_lines()[0].set_markerfacecolor('steelblue')
axes[0, 1].get_lines()[0].set_markeredgecolor('steelblue')
axes[0, 1].get_lines()[0].set_alpha(0.4)
axes[0, 1].set_title('B. Normal Q-Q Plot of Systolic Blood Pressure (mmHg)')
axes[0, 1].set_xlabel('Theoretical Normal Quantiles')
axes[0, 1].set_ylabel('Ordered Systolic BP Values (mmHg)')

# Panel C: BMI Histogram + Fitted Normal PDF
x_bmi = np.linspace(df_final['BMI'].min(), df_final['BMI'].max(), 300)
pdf_bmi = stats.norm.pdf(x_bmi, loc=bmi_mean, scale=bmi_sd)
sns.histplot(df_final['BMI'], bins=35, stat='density', color='peachpuff', edgecolor='black', alpha=0.6, ax=axes[1, 0], label='Empirical Density')
sns.kdeplot(df_final['BMI'], color='darkorange', linewidth=2, ax=axes[1, 0], label='Empirical KDE')
axes[1, 0].plot(x_bmi, pdf_bmi, 'r--', linewidth=2.2, label=f'Normal Fit N({bmi_mean:.1f}, {bmi_sd:.1f}^2)')
axes[1, 0].axvline(30, color='purple', linestyle=':', linewidth=2, label='Obesity Cutoff (30 kg/m^2)')
axes[1, 0].set_title(f'C. BMI Distribution vs. Normal Fit (Skew={bmi_skew:.2f}, Kurt={bmi_kurt:.2f})')
axes[1, 0].set_xlabel('Body Mass Index (kg/m^2)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# Panel D: Normal Q-Q Plot for BMI
stats.probplot(df_final['BMI'], dist="norm", plot=axes[1, 1])
axes[1, 1].get_lines()[0].set_markerfacecolor('darkorange')
axes[1, 1].get_lines()[0].set_markeredgecolor('darkorange')
axes[1, 1].get_lines()[0].set_alpha(0.4)
axes[1, 1].set_title('D. Normal Q-Q Plot of Body Mass Index (kg/m^2)')
axes[1, 1].set_xlabel('Theoretical Normal Quantiles')
axes[1, 1].set_ylabel('Ordered BMI Values (kg/m^2)')

plt.tight_layout()
plt.savefig(f'{fig_dir}/phase2_fig3.png')
plt.savefig('phase2_fig3.png')
plt.close()

# --- FIGURE 4: Two Random Variables, Covariance, and Survey Weights (Task 8 & Path A.4) ---
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel A: BMI vs Systolic BP with Mean Quadrant Lines
axes[0, 0].scatter(df_final['BMI'], df_final['Systolic_BP'], alpha=0.2, color='teal', s=15)
axes[0, 0].axvline(bmi_mean, color='red', linestyle='--', linewidth=1.5, label=f'E[BMI] = {bmi_mean:.2f}')
axes[0, 0].axhline(bp_mean, color='navy', linestyle='--', linewidth=1.5, label=f'E[SBP] = {bp_mean:.2f}')
axes[0, 0].set_title(f'A. Joint Scatter: BMI vs. Systolic BP (Cov = {cov_bp_bmi:+.2f})')
axes[0, 0].set_xlabel('BMI (kg/m^2)')
axes[0, 0].set_ylabel('Systolic BP (mmHg)')
axes[0, 0].legend(loc='upper right')

# Panel B: Age vs Systolic BP with Mean Quadrant Lines
age_mean = df_final['Age'].mean()
axes[0, 1].scatter(df_final['Age'], df_final['Systolic_BP'], alpha=0.2, color='slateblue', s=15)
axes[0, 1].axvline(age_mean, color='red', linestyle='--', linewidth=1.5, label=f'E[Age] = {age_mean:.2f}')
axes[0, 1].axhline(bp_mean, color='navy', linestyle='--', linewidth=1.5, label=f'E[SBP] = {bp_mean:.2f}')
axes[0, 1].set_title(f'B. Joint Scatter: Age vs. Systolic BP (Cov = {cov_bp_age:+.2f})')
axes[0, 1].set_xlabel('Age (Years)')
axes[0, 1].set_ylabel('Systolic BP (mmHg)')
axes[0, 1].legend(loc='upper left')

# Panel C: Sedentary Minutes vs Systolic BP
sed_mean = df_final['Sedentary_Min'].mean()
axes[1, 0].scatter(df_final['Sedentary_Min'], df_final['Systolic_BP'], alpha=0.2, color='darkgoldenrod', s=15)
axes[1, 0].axvline(sed_mean, color='red', linestyle='--', linewidth=1.5, label=f'E[Sedentary] = {sed_mean:.1f}m')
axes[1, 0].axhline(bp_mean, color='navy', linestyle='--', linewidth=1.5, label=f'E[SBP] = {bp_mean:.2f}')
axes[1, 0].set_title(f'C. Joint Scatter: Sedentary Time vs. Systolic BP (Cov = {cov_bp_sed:+.2f})')
axes[1, 0].set_xlabel('Daily Sedentary Behavior (Minutes/Day)')
axes[1, 0].set_ylabel('Systolic BP (mmHg)')
axes[1, 0].legend(loc='upper right')

# Panel D: Path A.4 Unweighted vs Survey-Weighted Empirical Probabilities
weight_comp_df = pd.DataFrame({
    'Event': ['High BP\n(SBP>=130)', 'Obesity\n(BMI>=30)', 'Short Sleep\n(Sleep<7h)', 'High Sedentary\n(Sed>=480m)', 'Cond. High BP\nP(H | Obese)'],
    'Unweighted (Sample Frequency)': [p_H, p_O, p_S, p_D, pair_HO['p_A_given_B']],
    'Weighted (Population Estimate)': [w_p_H, w_p_O, w_p_S, w_p_D, w_p_H_given_O]
}).set_index('Event')
weight_comp_df.plot(kind='bar', ax=axes[1, 1], color=['#fc8d62', '#66c2a5'], edgecolor='black', rot=0)
axes[1, 1].set_title('D. Unweighted vs. Survey-Weighted Empirical Probabilities [Path A.4]')
axes[1, 1].set_ylabel('Probability')
axes[1, 1].set_ylim(0, 0.55)
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.3f', padding=3, fontsize=8)

plt.tight_layout()
plt.savefig(f'{fig_dir}/phase2_fig4.png')
plt.savefig('phase2_fig4.png')
plt.close()

print("\n✅ All 4 Phase 2 figures generated and saved to 'figures/' and current directory!")


✅ All 4 Phase 2 figures generated and saved to 'figures/' and current directory!
